# NS-Spec v0.3 — Self-Verifying Notebook Specifications

**Status:** Approved design — supersedes v0.2  
**Primary solver:** Z3, reached *exclusively* through the existing SPUR `solve` MCP  
**Authoring and execution surface:** `.spec.ipynb` with native `ns-mermaid` cells  
**Portable artifact:** generated `.spec.md` snapshot (non-authoritative)  
**Review path:** retained as `ns-spec-v0.2-design.ipynb`; rename to v0.3 after spec review

> In v0.3, the notebook is the specification. Every formal unit is one
> executable `ns-mermaid` cell: its source visually declares topology,
> constraints, and named `@verify` points; running that same cell parses the
> declaration, lowers it to Canonical IR, verifies it through `solve`, and
> attaches proofs or counterexamples as cell outputs. No Python Z3 binding and
> no separately authored verification cell exist.

### Contents
1. Executive summary — notebook as spec
2. Same-cell architecture
3. Core principles
4. Per-cell verification decision flow
5. Determinism example in annotated Mermaid
6. Verification menu
7. Conformance-test generation
8. `solve` integration and cell ports
9. Lifecycle and statuses
10. NS-Mermaid grammar and milestones
11. Solver-substrate evidence and verifier caveats
12. Normative `ns-mermaid` cell contract, procedure, and adoption conditions

## 1. Executive summary

NS-Spec v0.3 makes a notebook cell the smallest authoritative specification
unit. An author writes annotated Mermaid; the same cell run renders the diagram,
parses its formal annotations, constructs Canonical IR, generates named proof
obligations, calls the existing `solve` MCP, and stores results in that cell's
outputs. A human approves the normalized meaning and fresh proof report before
implementation proceeds.

| Aspect | v0.2 | v0.3 |
|---|---|---|
| Authority | Markdown + `spec-yaml` | annotated Mermaid source in an `ns-mermaid` cell |
| Visualization | separate rendering | the declaration itself is the diagram |
| Verification | separately orchestrated audit | same-cell execution and outputs |
| Formal engine | `solve_constraints` / `solve_smt` | unchanged; still exclusive |
| Generated state | IR, obligations, reports | read-only cell outputs and typed ports |
| Portable Markdown | authorable profile | generated, non-authoritative snapshot |
| Acceptance | bounded conformance vs proof | unchanged, but scoped and hashed per cell |

### Why the same-cell model

A declaration and its verdict cannot drift into separate authored artifacts. The
cell source is pure and reviewable; execution never rewrites it. The cell runner
emits rendered Mermaid, normalized IR, individual obligation results, and an
aggregate proof report. Downstream report and conformance cells consume those
outputs through the reactive DAG.

## 2. Same-cell architecture

```mermaid
flowchart LR
    subgraph CELL[one executable ns-mermaid cell]
        SRC[annotated Mermaid source]
        PARSE[strict parser and type checker]
        IR[Canonical IR plus source map]
        PLAN[named proof obligations]
        OUT[cell outputs diagram IR proofs counterexamples]
        SRC --> PARSE --> IR --> PLAN
    end
    PLAN --> SOLVE((solve MCP Z3))
    SOLVE --> OUT
    OUT --> PORTS[typed ports spec_ir proof_report verified]
    PORTS --> REPORT[downstream notebook report and CI gate]
    EDIT[source or upstream input changes] -. invalidates and reruns .-> SRC
```

This is literally one notebook cell: one source record, one execution, and one
output bundle. The `solve` call happens inside the `ns-mermaid` cell runner. It
is not a Python kernel call and not a separately authored verification cell.
The cell may emit outputs and ports, but MUST NOT rewrite its own source, consume
its own emitted ports, or mutate any cell it depends on; those rules keep the
reactive graph acyclic.

The runner is kernel-independent and constrained. Its source MIME is
`application/vnd.spur.ns-mermaid+text`; its structured result MIME is
`application/vnd.spur.ns-proof+json`. The ordinary Mermaid renderer consumes the
same source and displays annotations as label text; the NS parser reads those
annotations from source, never from the rendered DOM.

## 3. Core principles

1. **The notebook is the specification.** `.spec.ipynb` is authoritative;
   generated Markdown is a review/export artifact only.
2. **One cell is one self-verifying formal unit.** Its Mermaid source, rendered
   diagram, Canonical IR, obligations, and verdict share one cell identity.
3. **Mermaid declares the semantics.** Stable graph IDs plus `@type`,
   `@input`, `@output`, `@state-var`, `@state`, `@transition`, `@branch`,
   `@requires`, `@guard`, `@update`, `@when`, `@ensures`, `@invariant`,
   `@verify`, and `@witness` annotations are the only authored formal source.
4. **Canonical IR is generated and auditable.** It is the verifier input, never
   a second authoring surface. Every IR element maps back to a cell and source
   span.
5. **Z3 is reached only through `solve`.** B′ is preferred; `solve_smt` is the
   guarded escape hatch. The notebook never imports or invokes Z3 directly.
6. **Source is pure; outputs are derived.** Running a cell may emit diagram,
   IR, proofs, models, diagnostics, and ports, but may not modify source.
7. **Verification is fail-closed.** Parse/type errors, missing obligations,
   stale results, `unknown`, `timeout`, unsupported terms, and internal errors
   all block approval.
8. **Proof meaning is explicit.** Proof obligations use assert-negation
   (`unsat` proves no counterexample); witness obligations expect `sat`.
9. **Bounded is not proven.** Generated conformance vectors yield
   `conformance-passed`, never `refinement-proven`.
10. **Lowering is part of the trusted base.** An `unsat` result is proof-grade
    only when the NS-Mermaid→IR→B′/SMT lowering is validated.

## 4. Per-cell verification decision flow

Every `ns-mermaid` cell runs this pipeline as one execution. A cell-level report
is published only after all declared verification points reach terminal statuses.

```mermaid
flowchart TD
    RUN([run ns-mermaid cell]) --> M{valid Mermaid plus NS annotations}
    M -- no --> PM([parse diagnostic with source span])
    M -- yes --> T{IDs references and expressions typed}
    T -- no --> TM([type diagnostic with source span])
    T -- yes --> IR[emit normalized Canonical IR and IR hash]
    IR --> V{"each required @verify point present"}
    V -- no --> MV([missing-obligation failure])
    V -- yes --> P[generate obligation and obligation hash]
    P --> L{B-prime expressible}
    L -->|yes| BC[solve_constraints]
    L -->|no, allowlisted SMT theory| SMT[solve_smt]
    L -->|no, unsupported theory| UT([unsupported-term failure])
    BC --> R{status matches obligation expectation}
    SMT --> R
    R -- yes --> PASS[attach proof or witness]
    R -- no sat counterexample --> FAIL[attach counterexample]
    R -- unknown timeout error --> INC([inconclusive fail closed])
    PASS --> MORE{"more @verify points"}
    FAIL --> MORE
    INC --> MORE
    MORE -- yes --> P
    MORE -- no --> AGG[emit aggregate proof_report and verified port]
```

The aggregate `verified` port is true only when parsing and typing succeeded,
all mandatory points were evaluated against the current source/IR hashes, and
each result matched its declared proof or witness expectation. Any source or
upstream-input edit immediately makes the prior output stale.

## 4b. Determinism declared and verified in one cell

The v0.1 transfer contract guarded success on the output itself. In v0.3 the
same mistake is visible in its Mermaid declaration point:

```mermaid
flowchart TD
    INPUT["`@spec TRANSFER-ORIGINAL
@type TransferStatus = enum[success, insufficient_balance]
@input source_balance: Int
@input target_balance: Int
@input amount: Int
@output status: TransferStatus
@output source_after: Int
@output target_after: Int
@requires PRE_SOURCE: source_balance >= 0
@requires PRE_TARGET: target_balance >= 0
@requires PRE_AMOUNT: amount > 0`"]

    INSUFFICIENT["`@branch INSUFFICIENT
@when status = insufficient_balance
@ensures STATUS_INSUFFICIENT: status = insufficient_balance
@ensures SOURCE_INSUFFICIENT: source_after = source_balance
@ensures TARGET_INSUFFICIENT: target_after = target_balance`"]

    SUCCESS["`@branch SUCCESS
@when status = success
@ensures STATUS_SUCCESS: status = success
@ensures SOURCE_SUCCESS: source_after = source_balance - amount
@ensures TARGET_SUCCESS: target_after = target_balance + amount`"]

    CHECK["`@verify DET_ORIGINAL: prove determinism`"]
    INPUT --> INSUFFICIENT --> CHECK
    INPUT --> SUCCESS --> CHECK
    CHECK --> V{{same-cell result: sat counterexample}}
```

Both `@when` clauses select an output value instead of partitioning inputs. The
generated determinism counterexample formula is `sat`, so the same cell reports
that one eligible input admits both unchanged/insufficient and transferred/success
results.

The corrected cell declares an input partition and its verification points:

```mermaid
flowchart TD
    CTX["`@spec TRANSFER-001
@type TransferStatus = enum[success, invalid_amount, insufficient_balance]
@input source_balance: Int
@input target_balance: Int
@input amount: Int
@output status: TransferStatus
@output source_after: Int
@output target_after: Int
@requires PRE_SOURCE: source_balance >= 0
@requires PRE_TARGET: target_balance >= 0`"]

    INVALID["`@branch INVALID
@when amount <= 0
@ensures STATUS_INVALID: status = invalid_amount
@ensures SOURCE_INVALID: source_after = source_balance
@ensures TARGET_INVALID: target_after = target_balance`"]

    INSUFFICIENT["`@branch INSUFFICIENT
@when amount > 0 and source_balance < amount
@ensures STATUS_INSUFFICIENT: status = insufficient_balance
@ensures SOURCE_INSUFFICIENT: source_after = source_balance
@ensures TARGET_INSUFFICIENT: target_after = target_balance`"]

    SUCCESS["`@branch SUCCESS
@when amount > 0 and source_balance >= amount
@ensures STATUS_SUCCESS: status = success
@ensures SOURCE_SUCCESS: source_after = source_balance - amount
@ensures TARGET_SUCCESS: target_after = target_balance + amount`"]

    CHECK["`@invariant NONNEG: source_after >= 0 and target_after >= 0
@invariant CONSERVE: source_after + target_after = source_balance + target_balance
@verify DET_TRANSFER: prove determinism
@verify STATUS_GAP: prove partition_coverage
@verify STATUS_OVERLAP: prove partition_exclusive
@verify EACH_STATUS: witness each status`"]

    CTX --> INVALID --> CHECK
    CTX --> INSUFFICIENT --> CHECK
    CTX --> SUCCESS --> CHECK
```

One run of this cell renders the diagram and evaluates all four named points.
The expected outputs are `unsat` for determinism, gap, and overlap
counterexample formulas, and `sat` for each requested status witness. Each result
records the declaration cell version, IR hash, obligation hash, and exact
annotation source span.

## 5. Verification menu — `@verify` points

| Declaration | Generated obligation | Expected status | Mismatch meaning |
|---|---|---|---|
| `@verify <point-id>: witness non_vacuity` | `SAT(assumptions ∧ requires)` | `sat` | `unsat` = vacuous cell |
| `@verify <point-id>: witness consistency` | `SAT(requires ∧ ensures ∧ invariants)` | `sat` | `unsat` = conflicting declarations |
| `@verify <point-id>: prove determinism` | `SAT(Pre ∧ Post(x,y1) ∧ Post(x,y2) ∧ y1≠y2)` | `unsat` | `sat` = under-determined outputs |
| `@verify <point-id>: prove partition_coverage` | `SAT(pre ∧ ¬(g1 ∨ … ∨ gn))` | `unsat` | `sat` = uncovered input region |
| `@verify <point-id>: prove partition_exclusive` | `SAT(pre ∧ ⋁i<j(gi ∧ gj))` | `unsat` | `sat` = overlapping branches |
| `@verify <point-id>: witness branch <branch-id>` | `SAT(pre ∧ posts ∧ branch)` | `sat` | `unsat` = dead branch |
| `@verify <point-id>: prove initiate <inv>` | `SAT(initial ∧ ¬invariant)` | `unsat` | `sat` = invalid initial state |
| `@verify <point-id>: prove preserve <inv> on <transition>` | `SAT(inv ∧ guard ∧ update ∧ ¬inv(next))` | `unsat` | `sat` = transition counterexample |
| `@verify <point-id>: bounded_reachability k=<n>` | explicit k-step unroll | declared `sat` or `unsat` | always labeled bounded |

**Constructive totality is a structural gate.** Every exhaustive input branch
must define every output with total, well-typed terms before solver execution.
Per-branch satisfiability is a reachability audit, not a proof of
`∀x∃y.Post(x,y)`.

The v0.3 expression grammar is QF_LIA + Bool + enum. Nonlinear multiplication,
division, modulo, unbounded quantifiers, and unsupported Mermaid constructs fail
with source-located diagnostics unless the verification point explicitly opts
into an allowed `solve_smt` theory. `unknown` and `timeout` never count as
proof.

## 6. Conformance generation from the same cell

Proof points audit the declaration; witness points can also emit bounded test
vectors. The `ns-mermaid` cell runner exposes these through a typed
`conformance_vectors` port without executing the implementation itself.

```mermaid
flowchart LR
    CELL[ns-mermaid cell source] --> IR[Canonical IR]
    IR --> W1["@verify EACH_STATUS: witness each status"]
    IR --> W2["@verify BOUNDARIES: witness boundary predicates"]
    IR --> W3["@verify INV_EDGES: witness invariant edges"]
    W1 --> SOLVE((solve MCP))
    W2 --> SOLVE
    W3 --> SOLVE
    SOLVE --> OUT[cell output proof report plus vectors]
    OUT --> PORT[conformance_vectors port]
    PORT --> TEST[language-specific generated tests]
    TEST --> IMPL[real implementation]
    IMPL --> RESULT[conformance report]
```

Boundary requests are explicit annotations such as `@witness amount = 0`,
`@witness amount = 1`, or `@witness source_balance = amount - 1`. A plain `sat`
model is arbitrary and MUST NOT be described as minimized or boundary-selected.

`conformance-passed` means the implementation matched the declared outputs on
the emitted finite vectors. It is distinct from a proof. A false acceptance is
still possible if the NS-Mermaid parser, lowering, oracle generation, wrapper,
or implementation adapter is wrong; those components require differential and
end-to-end tests.

## 7. Same-cell `solve` integration and ports

```mermaid
sequenceDiagram
    participant U as Author
    participant C as ns-mermaid cell runner
    participant S as solve MCP
    participant O as Same cell outputs
    participant D as Downstream report or CI

    U->>C: run annotated Mermaid source
    C->>C: parse type-check normalize and hash
    loop each @verify point
        C->>S: solve_constraints or solve_smt
        S-->>C: sat unsat unknown timeout plus model
    end
    C->>O: diagram plus IR plus proof_report plus diagnostics
    O-->>D: typed ports with source and obligation hashes
    U->>C: edit source
    C-->>O: mark prior outputs stale then rerun
```

### Required output bundle

- `diagram`: rendered Mermaid or a source-located render error.
- `spec_ir`: normalized Canonical IR plus schema version and IR hash.
- `obligations[]`: ID, kind, expected status, lowered form, and obligation hash.
- `results[]`: solver status, model/counterexample, duration, and optional
  repository-local `solve_id`.
- `proof_report`: aggregate pass/fail/inconclusive state.
- `verified`: true only for a fresh, complete, matching proof report.
- `conformance_vectors`: requested bounded witnesses, if any.

Canonical hashes cover schema-versioned semantic content only. Transient duration,
transport metadata, and `solve_id` are displayed but excluded, so UI and headless
runs can produce identical semantic hashes.

The cell source and outputs share one notebook cell ID but remain separate data.
The runner may emit the ports above; it MUST NOT write source or consume its own
ports. Persistence through `solve_id` is a handoff cache, not the durable source
of truth—the declaration, normalized IR hash, and proof report stay in the
notebook artifact.

## 8. Cell and notebook lifecycle

```mermaid
stateDiagram-v2
    [*] --> draft
    draft --> invalid: Mermaid parse or annotation error
    draft --> typed: parse schema and types pass
    typed --> verifying: obligations generated
    verifying --> verified: all point expectations match
    verifying --> failed: any proof or witness expectation mismatches
    verifying --> inconclusive: unknown timeout unsupported or internal error
    verified --> approved: human approves normalized IR and report
    approved --> implementation_pending
    implementation_pending --> conformance_passed: bounded tests green
    implementation_pending --> refinement_proven: optional sound implementation lowering
    verified --> stale: source or upstream input changes
    failed --> stale: source or upstream input changes
    inconclusive --> stale: source or upstream input changes
    approved --> stale: source or upstream input changes
    stale --> typed: rerun current cell
```

A notebook-level gate is the conjunction of its required cell gates. Every
formal cell must be fresh; optional explanatory Markdown cells do not
participate. Changing one `ns-mermaid` cell invalidates that cell and only its
reactive downstream consumers. Approval records the cell source hash, IR hash,
obligation hashes, and aggregate report hash.

## 9. NS-Mermaid profile and milestones

### 9.1 Normative declaration vocabulary

| Annotation | Meaning |
|---|---|
| `@spec <id>` | begins one cell-local specification namespace |
| `@type <name> = <type-expr>` | declares an alias over `Int`, `Bool`, or `enum[...]` |
| `@input <name>: <type>` | declares a universally quantified logical input |
| `@output <name>: <type>` | declares a result related to the logical inputs |
| `@state-var <name>: <type>` | declares mutable model state for transition systems |
| `@requires <id>: <expr>` | input assumption or precondition |
| `@state <id>` / Mermaid node ID | declares a stable control-state identity |
| `@transition <id>` | binds a stable transition ID to diagram topology |
| `@branch <id>` | binds a stable decision-branch ID to a Mermaid node |
| `@guard <expr>` | transition enabling predicate |
| `@update <state-var>' = <expr>` | next-state assignment |
| `@when <expr>` | input branch predicate |
| `@ensures <id>: <expr>` | output relation or postcondition |
| `@invariant <id>: <expr>` | named state/output invariant |
| `@verify <id>: <kind>` | names a proof or witness obligation |
| `@witness <expr>` | adds an explicit bounded-model predicate |

Annotations live visibly inside Mermaid Markdown-string node labels. The parser
reads source text, not rendered DOM. IDs are case-sensitive and unique within a
cell. References must resolve locally or through explicit typed input ports;
implicit notebook-global symbols are forbidden.

### 9.2 Revised milestones

1. **NS-Mermaid grammar and parser** — supported Mermaid subset, annotation
   lexer, stable IDs, source spans, expression grammar, and diagnostics.
2. **Native `ns-mermaid` cell runner** — kernel-independent execution, Mermaid
   rendering, same-cell outputs, staleness, cancellation, and typed ports.
3. **Canonical IR and trusted lowering** — normalization plus B′/SMT generation,
   differential fixtures, round-trip checks, and mutation tests against false
   `unsat` results.
4. **Verification planner** — named proof/witness expansion, expectation
   matching, counterexample mapping, and aggregate cell reports.
5. **Conformance vector emitter** — explicit boundaries and witnesses into
   language-specific black-box tests.
6. **Headless notebook gate** — deterministic reactive execution, output-hash
   validation, CI reporting, and approval records.

Markdown `.spec.md` becomes a generated snapshot. It may show the Mermaid source,
normalized IR, and proof report, but editing it never changes the authoritative
notebook.

## 10. Solver-substrate evidence and verifier caveats

These runs validate the proof obligations and existing `solve` substrate; they
predate and therefore do not validate the native `ns-mermaid` cell runner.

All runs were executed on 2026-07-29 through the existing `solve_constraints`
MCP tool using B′ only. Runs #2–8 included the transfer input requirements,
postcondition implications, non-negative output invariants, and balance
conservation; run #1 intentionally encoded a mutant that violates them. `sat`
models below are concrete witnesses, not golden assignments;
Z3 may return different witnesses for the same satisfiable predicate. Transient
wall-clock timings are intentionally omitted.

| # | Check | Encoding (B′) | Reproduced result |
|---|---|---|---|
| 1 | Seeded mutation witness | insufficient input plus a mutant that returns `success` and applies the transfer | **sat** → `src=6,tgt=0,amt=8,status=success,sa=-2,ta=8` |
| 2 | **Determinism, ORIGINAL contract** | `pre ∧ Post(x,y1) ∧ Post(x,y2) ∧ y1≠y2` | **sat** → at `src=2,tgt=1,amt=1`, `insufficient`/unchanged and `success`/transferred are both valid |
| 3 | **Determinism, CORRECTED contract** | same counterexample formula with input-guarded posts | **unsat** → no two distinct outputs exist |
| 4 | Status-partition gap | `pre ∧ ¬(g_invalid ∨ g_insufficient ∨ g_success)` | **unsat** → guards are exhaustive |
| 5 | Status-partition overlap | `pre ∧ ⋁i<j(gi ∧ gj)` | **unsat** → guards are mutually exclusive |
| 6 | Conformance witness — `success` | `pre ∧ corrected_contract ∧ status=success` | **sat** → `src=7,tgt=7,amt=5 ⇒ sa=2,ta=12` |
| 7 | Conformance witness — `invalid_amount` | `pre ∧ corrected_contract ∧ status=invalid_amount` | **sat** → `src=3,tgt=0,amt=-8 ⇒ unchanged` |
| 8 | Conformance witness — `insufficient_balance` | `pre ∧ corrected_contract ∧ status=insufficient_balance` | **sat** → `src=3,tgt=0,amt=8 ⇒ unchanged` |
| 9 | SHIP preservation, original guard | `inv ∧ status=paid ∧ ship_update ∧ ¬inv(next)` | **sat** → `captured=0,next_status=shipped` |
| 10 | SHIP preservation, strengthened guard | run #9 plus `captured>0` | **unsat** → strengthened transition is inductive |

### What this proves

- Runs #2–3 reproduce the determinism defect and prove the corrected encoded
  relation deterministic via the assert-negation pattern.
- Runs #4–5 separately prove status-partition coverage and exclusivity; neither
  result is a substitute for full relation totality.
- Runs #6–8 establish that every declared transfer status has at least one
  witness suitable for a generated test. They remain bounded examples, not a
  refinement proof.
- Run #1 is an explicit seeded mutation, not symbolic inspection of a real
  implementation. It demonstrates a test-generation target without reviving the
  deleted implementation adapter.
- Runs #9–10 reproduce the distinction between invariant preservation and
  reachability discussed in §10.2 C2.

### Bottom line

The solver results validate the logical correction and the existing `solve` MCP
integration. They do **not** validate the not-yet-implemented NS-Mermaid cell runner,
NS-Mermaid→IR lowering, or generated-test runner; those components require their own
property-preserving and end-to-end conformance tests.

---

### 10.1 Grounding against `crates/spur-solver`

The solver-integration claims were checked against the real solver source via code-explore, and the unsat-core gap was confirmed by execution:

| Claim | Source | Status |
|---|---|---|
| B′ `Variable` / `ConstraintExpr` / `ConstraintOp` grammar (vars, tagged expr, ops, no `div`) | `spur-solver/src/types.rs:30-179` | exact match |
| Status envelope `Sat` / `Unsat` / `Unknown` / `Timeout` (+`Error`) | `types.rs:320-334` | match |
| Encoder hardcodes `(set-logic QF_NIA)` — nonlinear `mul` supported | `encode.rs:125` | reframes QF_LIA as a self-imposed fragment |
| SMT gate allowlist = `set-logic` / `assert` / `check-sat` / `get-model` / `get-value` / `push` / `pop` / `declare-*` | `smt_gate.rs:143-153` | **`get-unsat-core` REJECTED** → client-side subset-iteration |
| `unknown` never collapsed to `unsat` | test `fake_solver_unknown_is_not_collapsed_into_unsat` | enforced in code |
| `persist` → `solve_id`, worker-side solve tools | `spur-core/src/worker_server.rs`, test `solve_constraints_persists_when_requested` | wired + tested |

The `get-unsat-core` rejection was reproduced live: a `solve_smt` script containing it returns `command get-unsat-core at byte 71 is not allowed`. Net: the v0.3 cell runner can remain a **pure client of the existing `solve` MCP with zero changes to `spur-solver`**; widening the gate for native unsat-cores is a future optimization, not a requirement.

---

### 10.2 Verify-the-verifier corrections

**C1. The lowering is the #1 soundness risk (false unsat).** During evaluation, a mis-nested disjunction in a B-prime encoding produced three spurious `unsat` results — each indistinguishable from a genuine proof. The solver was correct (a minimal `sat` probe confirmed the engine and enum encoding were flawless); the bug was purely in the lowering. Lesson: a verdict is only as sound as the NS-Mermaid→IR→B′ lowering layer. The rule `unknown is not success` must extend to: an `unsat` produced from an untrusted lowering is not a proof until the lowering itself is validated — a lowering bug silently mints false proofs, strictly worse than `unknown`. Implication: the lowering layer needs round-trip or property-preserving checks (or a small trusted kernel); this is the top engineering risk for an NS-Spec implementation, ahead of headless-CI and executable-cell security.

**C2. Label §16.7 preservation vs §16.8 reachability distinctly (SHIP downgrade).** The §16.7 invariant-preservation check runs over all invariant-satisfying states, not just reachable ones. For ORDER-LIFECYCLE it returns `sat` (counterexample `captured=0`): `INV-SHIPPED-PAID` is non-inductive, because SHIP from `paid` does not require `captured>0`. This is a code smell, not a runtime defect — the violating state `paid` with `captured=0` is unreachable (the only transition into `paid` is PAY, which sets `captured=amount>0`). §16.7 and §16.8 return different verdicts by design; a spec can fail §16.7 yet be safe. Rule: the proof report MUST label which check produced a verdict, and never present a §16.7 `sat` as unsafe without a reachability verdict. (Reachability here is by inspection of the transition relation; a solver proof needs a correct unrolling — which is easy to mis-encode, see C1.)

---

## 11. Normative v0.3 decision — one Mermaid declaration, one cell, one verdict

**Decision.** The authoritative spec format is `.spec.ipynb`. Each formal cell
has native type `ns-mermaid`. Its single source is annotated Mermaid and its
single execution performs both declaration processing and verification through
`solve`. The same cell owns the rendered diagram, normalized IR, named proof
results, counterexamples, and typed output ports.

```mermaid
flowchart TD
    subgraph ONE[one ns-mermaid notebook cell]
        SOURCE["`SOURCE
Mermaid topology
@type @input @output @state-var
@branch @guard @update
@invariant @verify`"]
        RUN["`RUNNER
parse and type
normalize and hash
generate obligations`"]
        OUTPUT["`OUTPUTS
rendered diagram
Canonical IR
proofs or counterexamples
verified port`"]
        SOURCE --> RUN --> OUTPUT
    end
    RUN --> SOLVE((solve MCP and Z3))
    SOLVE --> OUTPUT
    OUTPUT --> DOWNSTREAM[report conformance and CI cells]
```

### Same-cell means source plus outputs, not self-mutation

A notebook code cell already separates immutable source from derived outputs.
`ns-mermaid` uses that model without a general-purpose language kernel. The
runner reads its source and explicit upstream ports, then replaces only its
outputs. It never writes source, reads its prior output as an input, or mutates a
cell it depends on. Consequently, one cell can self-verify without a graph
feedback edge.

### Sole authored declaration language

There is no separate typed expression block and no hand-authored Z3 cell.
`@type`, `@input`, `@output`, `@state-var`, `@state`, `@transition`,
`@branch`, `@requires`, `@guard`, `@update`, `@when`, `@ensures`,
`@invariant`, `@verify`, and `@witness` annotations are part of the strict
NS-Mermaid source.
Canonical IR is a generated, auditable output. Z3 sees only the B′/SMT lowering
of that IR through the existing `solve` service.

Generated `.spec.md` is a portable review snapshot, not an independently
editable profile. Import or round-trip authoring is outside the v0.3 MVP because
it would reintroduce two sources of truth.

### 11.1 Cell authoring and execution procedure

1. **Create one `ns-mermaid` cell** and assign a stable cell ID.
2. **Declare the visual topology** with explicit Mermaid node IDs and unique
   transition/branch labels.
3. **Declare formal meaning in the same source** using visible `@type`,
   `@input`, `@output`, `@state-var`, control IDs, predicates, updates,
   postconditions, and invariants.
4. **Name every required check** with `@verify`; add explicit `@witness`
   predicates for boundary or conformance models.
5. **Run that same cell.** The native runner parses Mermaid and annotations,
   resolves references, type-checks expressions, and records source spans.
6. **Inspect normalized IR output.** Verification cannot be approved until the
   human-visible IR matches the intended diagram semantics.
7. **Generate obligations.** Each named point receives an expectation, lowered
   expression, source hash, IR hash, and obligation hash.
8. **Verify through `solve`.** B′ goes to `solve_constraints`; an explicitly
   allowed theory goes to `solve_smt`. The notebook never invokes Z3 directly.
9. **Render same-cell results.** Proof badges appear next to declaration points;
   `sat` proof failures include counterexamples mapped back to annotations.
10. **Emit typed ports.** Fresh IR, proof report, verified state, and optional
    conformance vectors flow to downstream report/CI cells.
11. **Invalidate on change.** Editing source or an upstream typed input marks all
    prior outputs stale before reactive re-execution.
12. **Approve or block.** Approval requires a fresh normalized IR review and all
    mandatory point expectations to match.

### 11.2 Failure semantics

| Condition | Same-cell outcome |
|---|---|
| Mermaid render error | source-located parse failure; no solver call |
| Valid Mermaid but invalid NS annotation | profile diagnostic; no solver call |
| Unresolved ID or type mismatch | typed diagnostic; no solver call |
| Unsupported expression/theory | `unsupported-term`; fail closed |
| Proof obligation returns `sat` | refuted with concrete counterexample |
| Witness obligation returns `unsat` | requested branch/model is unreachable |
| `unknown`, `timeout`, or solver error | inconclusive; fail closed |
| Source/IR/obligation hash mismatch | stale; result cannot approve |
| Some points pass and one fails | retain all point results; aggregate fails |
| Cell run is cancelled | partial results marked incomplete and non-approving |

### 11.3 Adoption conditions and test strategy

1. **Native constrained runner.** Add an `ns-mermaid` cell type rather than
   executing Python or arbitrary JavaScript. Enforce source-size, AST-size,
   obligation-count, solver-time, and output-size limits.
2. **Headless parity.** The same cell runner and reactive semantics must execute
   without a window for CI; UI and headless runs must produce identical hashes.
3. **Trusted lowering evidence.** Golden parser fixtures, source-span tests,
   normalized-IR snapshots, B′/SMT differential tests, and mutation tests must
   detect dropped, inverted, duplicated, or mis-nested clauses.
4. **Same-cell integration tests.** Editing a declaration must stale its outputs,
   rerun its proofs, replace only outputs, and rerun only downstream consumers.
5. **Status tests.** Exercise `sat`, `unsat`, `unknown`, `timeout`, unsupported,
   cancellation, mixed multi-point results, and counterexample source mapping.
6. **Security tests.** Mermaid and annotation input must never escape into shell,
   Python, raw Z3 argv, filesystem access, or uncontrolled network calls.
7. **Artifact tests.** Saving/reopening preserves source and outputs; approval is
   rejected when persisted hashes no longer match current source.

**Top risk: lowering soundness.** A buggy NS-Mermaid→IR→B′/SMT lowering can mint
false `unsat` proofs. The lowering belongs to the trusted base and must be small,
versioned, auditable, and tested independently of Z3.

`ns-spec.spec.ipynb` remains useful prototype evidence for the logical examples,
but its hand-authored Python/Z3 cells are **not** a conforming v0.3 reference.
The first conforming reference requires the native same-cell runner described
above.